# 02 Preprocessing and Splitting

This notebook creates the modeling table, converts ratings into implicit feedback, performs a temporal leave-one-out split, samples training negatives, and creates fixed validation/test candidate sets.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config
from src.data_loader import load_movielens_100k
from src.preprocessing import merge_dataset, encode_categorical_features, temporal_leave_one_out_split, build_movie_metadata
from src.negative_sampling import build_user_positive_item_dict, sample_train_negatives, create_eval_candidates
from src.utils import ensure_dir, save_json, save_dataframe, set_seed

set_seed()
ensure_dir(config.PROCESSED_DATA_DIR)

## Build Processed Interaction Table

Ratings greater than or equal to 4 are treated as positive implicit feedback. Lower ratings are kept in the processed interaction file for analysis, but positive-only data is used for leave-one-out ranking splits.

In [ ]:
ratings, users, movies = load_movielens_100k(config.RAW_DATA_DIR)
interactions = merge_dataset(ratings, users, movies)
interactions, mappings, metadata = encode_categorical_features(interactions)

save_dataframe(interactions, config.PROCESSED_DATA_DIR / 'interactions_processed.csv')
save_json(mappings, config.PROCESSED_DATA_DIR / 'mappings.json')

display(interactions.head())
print('Processed interactions:', interactions.shape)

## Temporal Leave-One-Out Split

For each eligible user, the newest positive interaction becomes test, the second newest becomes validation, and earlier positives become training. This is preferable to a random split because it avoids training on future interactions to predict earlier behavior.

In [ ]:
train_pos, val_pos, test_pos = temporal_leave_one_out_split(interactions)
print(train_pos.shape, val_pos.shape, test_pos.shape)
save_dataframe(val_pos, config.PROCESSED_DATA_DIR / 'validation_positive.csv')
save_dataframe(test_pos, config.PROCESSED_DATA_DIR / 'test_positive.csv')

## Negative Sampling

NCF is trained with positive interactions and sampled unobserved items as negatives. For negative rows, user and time context are copied from the positive event, while movie context is replaced with the sampled movie metadata. Validation and test use fixed candidate sets so all models are compared on the same ranking task.

In [ ]:
movie_metadata = build_movie_metadata(interactions)
all_movie_indices = np.array(sorted(interactions['movie_idx'].unique()))
user_pos_dict = build_user_positive_item_dict(interactions)

train = sample_train_negatives(
    train_pos, all_movie_indices, user_pos_dict, config.NUM_NEGATIVES_TRAIN, movie_metadata
)
validation = sample_train_negatives(
    val_pos, all_movie_indices, user_pos_dict, config.NUM_NEGATIVES_TRAIN, movie_metadata, seed=config.RANDOM_SEED + 1
)
test = test_pos.copy()

val_candidates = create_eval_candidates(
    val_pos, all_movie_indices, user_pos_dict, config.NUM_NEGATIVES_EVAL, movie_metadata, seed=config.RANDOM_SEED + 2
)
test_candidates = create_eval_candidates(
    test_pos, all_movie_indices, user_pos_dict, config.NUM_NEGATIVES_EVAL, movie_metadata, seed=config.RANDOM_SEED + 3
)

save_dataframe(train, config.PROCESSED_DATA_DIR / 'train.csv')
save_dataframe(validation, config.PROCESSED_DATA_DIR / 'validation.csv')
save_dataframe(test, config.PROCESSED_DATA_DIR / 'test.csv')
save_dataframe(val_candidates, config.PROCESSED_DATA_DIR / 'val_candidates.csv')
save_dataframe(test_candidates, config.PROCESSED_DATA_DIR / 'test_candidates.csv')

print('train:', train.shape)
print('validation:', validation.shape)
print('test positives:', test.shape)
print('validation candidates:', val_candidates.shape)
print('test candidates:', test_candidates.shape)

The saved files in `data/processed/` are the reproducible inputs for all later notebooks.